# CIC-UNSW-NB15 vs. UNSW-NB15 — Colab runbook

End-to-end reproduction of the experiments in *Enhancing Intrusion Detection Performance: A Comparative Study of Machine Learning on CIC-UNSW-NB15 vs. UNSW-NB15*.

Run the cells in order. Do not skip cell 3: it is where the identifier columns are identified, and every later cell depends on getting that right.

| Cell | Step | Time |
|---|---|---|
| 1 | Clone the repository and install dependencies | ~2 min |
| 2 | Locate the datasets | seconds |
| 3 | **Inspect the headers** — defines the columns to drop | ~1 min |
| 4 | Write `config.json` | seconds |
| 5 | Smoke test — validates the whole path cheaply | 3–5 min |
| 6 | Main run | 60–90 min |
| 7 | Ablation run | 20–30 min |
| 8 | Significance tests and figures | ~2 min |

**Before starting**, download the two datasets and place them in a Google Drive folder. They are not distributed with this repository:

- UNSW-NB15 — https://research.unsw.edu.au/projects/unsw-nb15-dataset
- CIC-UNSW-NB15 — https://www.unb.ca/cic/datasets/

A free Colab session disconnects after roughly 90 minutes idle and 12 hours maximum, and `/content` is wiped when it does. Keep the data in Drive and write the results to Drive, not to session storage.

In [ ]:
# ===== 1. Clone the repository and install dependencies =====
REPO = 'https://github.com/USERNAME/cic-unsw-nb15-comparison.git'   # <-- your fork or clone URL

!git clone -q {REPO} /content/repo || echo "clone failed -- check the URL"
%cd /content/repo
!pip -q install -r requirements.txt

import sklearn, xgboost, pandas as pd, numpy as np, scipy
print('sklearn', sklearn.__version__, '| xgboost', xgboost.__version__,
      '| pandas', pd.__version__, '| scipy', scipy.__version__)
print('\nRecord these versions: results are reproducible only against them.')

In [ ]:
# ===== 2. Locate the datasets =====
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Adjust to wherever you placed the CSVs
DATA    = Path('/content/drive/MyDrive/ids_paper/data')
RESULTS = Path('/content/drive/MyDrive/ids_paper/results')   # written to Drive so it survives a disconnect
RESULTS.mkdir(parents=True, exist_ok=True)

CIC_CSV  = DATA / 'CIC_NB15.csv'
UNSW_CSV = DATA / 'unsw_all.csv'      # see docs/reproduce.md section 1 to build this

for p in (CIC_CSV, UNSW_CSV):
    print(('OK      ' if p.exists() else 'MISSING '), p)

### Building `unsw_all.csv`

The pipeline performs its own repeated stratified splitting, so it needs the UNSW-NB15 partitions pooled into one file. Run this once if you have not already:

```python
a = pd.read_csv(DATA / 'UNSW_NB15_training-set.csv', low_memory=False)
b = pd.read_csv(DATA / 'UNSW_NB15_testing-set.csv',  low_memory=False)
assert list(a.columns) == list(b.columns)
pd.concat([a, b], ignore_index=True).to_csv(DATA / 'unsw_all.csv', index=False)
print(len(a), '+', len(b), '=', len(a) + len(b))     # 175,341 + 82,332 = 257,673
```

Pooling raises accuracy relative to the prescribed split, because the prescribed partition is deliberately constructed to be difficult. Section 4.9 of the paper quantifies this. The absolute figures are therefore not comparable with prescribed-partition studies; the matched-condition analysis is unaffected, since it depends on differences between the datasets rather than on absolute values.

### 3. Inspect the headers — the decisive step

This cell determines three things that cannot be guessed:

1. The name of the label column in each file.
2. Which identifier columns must be dropped. Any IP address, port, flow ID or timestamp left in the feature matrix lets the model memorise which hosts generated the attacks instead of learning traffic behaviour. **This is the usual cause of implausible scores above 99%.**
3. The actual CICFlowMeter column names, needed for the E2 feature map — they vary between tool versions.

In [ ]:
# ===== 3. Inspect the headers =====
import re
import numpy as np
import pandas as pd

SUSPECT = re.compile(r'ip|port|flow.?id|timestamp|stime|ltime|^id$|^no$|index|unnamed',
                     re.IGNORECASE)

def inspect(path, name, nrows=200_000):
    print('=' * 78); print(name); print('=' * 78)
    df = pd.read_csv(path, low_memory=False, nrows=nrows)
    df.columns = [str(c).strip() for c in df.columns]
    print(f'first {len(df):,} rows | {df.shape[1]} columns\n')

    print('--- candidate label columns:')
    for c in df.columns:
        if c.strip().lower() in {'label', 'class', 'attack', 'attack_cat', 'category'}:
            print(f'    {c!r}  ->  {df[c].value_counts().head(4).to_dict()}')

    print('\n--- identifier-like columns (copy these into extra_leakage_columns):')
    hits = [c for c in df.columns if SUSPECT.search(c)]
    for c in hits:
        print(f'    {c!r}   (distinct values: {df[c].nunique()})')
    if not hits:
        print('    none found')

    num = df.select_dtypes(include=[np.number])
    print(f'\n--- columns containing inf: {[c for c in num.columns if np.isinf(df[c]).any()] or "none"}')
    print(f'--- constant columns: {[c for c in df.columns if df[c].nunique(dropna=False) <= 1] or "none"}')
    print(f'--- rows with NaN: {int(df.isna().any(axis=1).sum()):,}')
    print(f'\n--- all columns:\n{list(df.columns)}\n')
    return df

cic_head  = inspect(CIC_CSV,  'CIC-UNSW-NB15')
unsw_head = inspect(UNSW_CSV, 'UNSW-NB15')

### 4. Configuration

Fill both lists from the output of cell 3.

- `extra_leakage_columns`: every identifier column you saw. **Over-inclusion is harmless; under-inclusion invalidates the results.**
- `common_feature_map`: key = column name in UNSW-NB15, value = its semantic counterpart in CIC-UNSW-NB15. Drop any pair whose two sides are not both present.

The values below are those used for the published results.

In [ ]:
# ===== 4. Write config.json =====
import json

CONFIG = {
    # The public releases have already had their five-tuple identifiers removed;
    # 'id' and 'attack_cat' are handled by LEAKAGE_COLUMNS in the script itself.
    "extra_leakage_columns": [],

    # 12 semantically aligned pairs -- key: UNSW-NB15, value: CIC-UNSW-NB15
    "common_feature_map": {
        "dur":    "Flow Duration",
        "spkts":  "Total Fwd Packet",
        "dpkts":  "Total Bwd packets",
        "sbytes": "Total Length of Fwd Packet",
        "dbytes": "Total Length of Bwd Packet",
        "smean":  "Fwd Packet Length Mean",
        "dmean":  "Bwd Packet Length Mean",
        "rate":   "Flow Packets/s",
        "sinpkt": "Fwd IAT Mean",
        "dinpkt": "Bwd IAT Mean",
        "swin":   "FWD Init Win Bytes",
        "dwin":   "Bwd Init Win Bytes",
    },
}

CFG = Path('/content/repo/config/config.json')
CFG.write_text(json.dumps(CONFIG, indent=2))

# verify both sides of every pair exist before running anything expensive
lo_u = {c.strip().lower(): c for c in unsw_head.columns}
lo_c = {c.strip().lower(): c for c in cic_head.columns}
ok = 0
for u, c in CONFIG['common_feature_map'].items():
    if u.lower() in lo_u and c.lower() in lo_c:
        ok += 1
    else:
        print(f'  unmatched: {u!r} in UNSW = {u.lower() in lo_u} | '
              f'{c!r} in CIC = {c.lower() in lo_c}')
print(f'\nvalid pairs: {ok} of {len(CONFIG["common_feature_map"])}')
print('E2 is not meaningful below about five valid pairs.')

### 5. Smoke test

Two classifiers, 20,000 records, one repetition. The point is not the numbers but confirming the whole path works before spending an hour. **Do not use these numbers for anything.**

In [ ]:
# ===== 5. Smoke test =====
!python src/ids_evaluation.py \
    --cic "{CIC_CSV}" --unsw "{UNSW_CSV}" \
    --label-cic Label --label-unsw label \
    --config config/config.json \
    --outdir "{RESULTS}/smoke" \
    --sample 20000 --models RF,DT --n-repeats 1 --skip-shap

print(pd.read_csv(RESULTS / 'smoke' / 'dataset_summary.csv', index_col=0))

**Check three things before continuing:**

- `dropped_identifier_columns` contains everything you listed in cell 4. If it is empty when it should not be, go back.
- `malicious_pct` is plausible for each file. Deduplication shifts it, so it will not equal the dataset's headline ratio.
- Accuracy is around 0.97–0.98 for CIC-UNSW-NB15 and 0.89–0.92 for UNSW-NB15. **A Decision Tree scoring above 0.99 on 20,000 records is a sign of a remaining identifier column, not of success.**

---

### 6. Main run

`--sample` is the main trade-off. K-Nearest Neighbours dominates the runtime, because its inference cost grows with training-set size and dimensionality, and the CICFlowMeter space has 67 dimensions after preprocessing.

| Sample | Approximate time (5 models, 3 × 5-fold) |
|---|---|
| 50,000 | ~35 min |
| 100,000 | **60–90 min (published configuration)** |
| 200,000+ | several hours; risks the session limit |

Note that the computation runs three times — baseline, E1 and E2 — so the total is roughly triple a single evaluation.

`--sample` draws a uniform random subsample and is safe for reported experiments. `--nrows` takes a non-random head slice and is for smoke tests only.

Keep the browser tab open and revisit it every fifteen minutes or so; an idle session is disconnected.

In [ ]:
# ===== 6. Main run =====
SAMPLE = 100_000
MODELS = 'RF,XGB,AdaBoost,DT,KNN'

!python src/ids_evaluation.py \
    --cic "{CIC_CSV}" --unsw "{UNSW_CSV}" \
    --label-cic Label --label-unsw label \
    --config config/config.json \
    --outdir "{RESULTS}/main" \
    --sample {SAMPLE} --models {MODELS} \
    --n-splits 5 --n-repeats 3

### 7. Ablation run

Repeats the protocol with the highest-attribution features removed: `Bwd Packet Length Min` and `Fwd Seg Size Min` from CIC-UNSW-NB15, and `sttl`, `ct_state_ttl` and `service` from UNSW-NB15. TTL-derived attributes reflect the traffic-generation environment rather than intrinsic flow behaviour, so they are a plausible source of optimistic bias.

Expect no meaningful change. That is the finding: gain-based importance in these datasets does not indicate necessity.

In [ ]:
# ===== 7. Ablation run =====
!python src/ids_evaluation.py \
    --cic "{CIC_CSV}" --unsw "{UNSW_CSV}" \
    --label-cic Label --label-unsw label \
    --config config/config_ablation.json \
    --outdir "{RESULTS}/ablation" \
    --sample {SAMPLE} --models RF,XGB,DT \
    --n-splits 5 --n-repeats 3

In [ ]:
# ===== 8. Significance tests and figures =====
!python src/significance_tests.py --results "{RESULTS}/main" --model XGB \
    --out "{RESULTS}/main/dataset_significance.csv"

!python src/make_figures.py --results "{RESULTS}/main" --outdir "{RESULTS}/figures"
!python src/make_methodology_figure.py

# collect everything into one workbook
sheets, seen = {}, []
for folder in ['main', 'ablation']:
    for csv in sorted((RESULTS / folder).glob('*.csv')):
        name = f'{folder}_{csv.stem}'[:31]
        while name in seen:
            name = name[:29] + '_2'
        seen.append(name)
        sheets[name] = pd.read_csv(csv)

out = RESULTS / 'paper_tables.xlsx'
with pd.ExcelWriter(out) as writer:
    for name, df in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)
print(f'{len(sheets)} sheets -> {out}')

### Mapping the output to the paper

| Paper | File |
|---|---|
| Table 2 (dataset characteristics) | `main/dataset_summary.csv` |
| Tables 4 and 5 (per-dataset results) | `main/main_results_CIC.csv`, `main_results_UNSW.csv` |
| Table 6 (McNemar) | `main/mcnemar_CIC.csv`, `mcnemar_UNSW.csv` |
| Table 7 (matched conditions) | `main/matched_E1.csv`, `matched_E2.csv` |
| Table 9 (ablation) | `ablation/main_results_*.csv` |
| Table 10 (cost) | `main/cost_CIC.csv` |
| Table 12 (significance) | `main/dataset_significance.csv` |
| Figures 2–8 | `figures/` |
| Per-fold raw scores | `main/folds_*.csv` |

The `folds_*.csv` files hold the fifteen individual fold scores behind every mean and standard deviation, and are the source for the error bars and for all significance testing.

**Download the results folder when the run finishes.** Drive keeps the files, but a session that disconnects mid-run leaves partial results without warning.

See `docs/reproduce.md` in the repository for the known limitations of this reproduction: hyper-parameters are fixed rather than tuned, experiment E3 is not reproducible from the public releases, and the fold scores are not mutually independent.